In [ ]:
import random
import numpy as np
import torch
import monai
 
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    monai.utils.set_determinism(seed=seed)
 
set_seed(42)
 
import torch
import os
import json
from monai.data import ImageDataset, DataLoader
from monai.transforms import (
    Compose, EnsureChannelFirst, Resize,
    ScaleIntensity, NormalizeIntensity,
    RandAffine, RandGaussianNoise
)
from torch.utils.tensorboard import SummaryWriter
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
# ------------------- Enhancement+Normalization -------------------
train_transforms = Compose([
    EnsureChannelFirst(),
    Resize((96, 96, 96)),
    ScaleIntensity(minv=0.0, maxv=1.0),
    NormalizeIntensity(nonzero=True),
    RandAffine(prob=0.2, rotate_range=(-2,2), translate_range=0.02),
    RandGaussianNoise(prob=0.1, std=0.01),
])
 
val_transforms = Compose([
    EnsureChannelFirst(),
    Resize((96, 96, 96)),
    ScaleIntensity(minv=0.0, maxv=1.0),
    NormalizeIntensity(nonzero=True),
])
 
## 1. Load the locked-in splits
load_path = os.path.expanduser('~/Desktop/brain-math/deeplearn/GLM/GLM_kfold_splits.json')
with open(load_path, 'r') as f:
    saved_splits = json.load(f)
 
current_fold = "fold_1"
fold_data = saved_splits[current_fold]
 
train_images = fold_data["train_images"]
val_images = fold_data["val_images"]
train_labels = torch.as_tensor(fold_data["train_labels"], dtype=torch.long)
val_labels = torch.as_tensor(fold_data["val_labels"], dtype=torch.long)
 
train_ds = ImageDataset(image_files=train_images, labels=train_labels, transform=train_transforms)
val_ds = ImageDataset(image_files=val_images, labels=val_labels, transform=val_transforms)
 
batch_size = 4
num_workers = 2
 
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
 
# ------------------- Model: Lightweight Classifier (without pre training) -------------------
model = monai.networks.nets.Classifier(
    in_shape=(1, 96, 96, 96),  # (channels, D, H, W)
    classes=2,                 # Replaces out_classes
    channels=(16, 32, 64),
    strides=(2, 2)             # Length should typically be len(channels) - 1
).to(device)
 
loss_function = torch.nn.CrossEntropyLoss()
 
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
#Learning rate decay: The validation set automatically decreases without increasing
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6)
 
# ------------------- training -------------------
best_metric = -1
patience = 10
patience_counter = 0
max_epochs = 80
writer = SummaryWriter()
 
for epoch in range(max_epochs):
    print("-" * 30)
    print(f"Epoch {epoch+1}/{max_epochs}")
    model.train()
    train_loss = 0
    step = 0
 
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_function(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        step += 1
 
    avg_train_loss = train_loss / step
    print(f"Train loss: {avg_train_loss:.4f}")
 
    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            val_loss += loss_function(out, y).item()
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += len(y)
 
    val_acc = correct / total
    scheduler.step(val_acc)
    print(f"Val acc: {val_acc:.4f}")
 
    if val_acc > best_metric:
        best_metric = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_classifier.pth")
        print("saved best metric model")
    else:
        patience_counter += 1
        print(f"No improvement: {patience_counter}/{patience}")
 
    if patience_counter >= patience:
        print("\nEarly stopping")
        break
 
print(f"Best accuracy: {best_metric:.4f}")
writer.close()
 
 